In [1]:
# ============================================================================
# NB07 - Camada Generativa: Reescrita de Descrições com LLM Local
# ============================================================================
# Objetivo: usar gemma3:4b (via Ollama) para reescrever descrições dos
# animais do tier High da priority queue do NB06, tornando-as mais apelativas
# para potenciais adotantes — sem inventar factos.
#
# Input principal: results/nb06_multimodal/priority_queue.parquet
# Output principal: results/nb07_llm_rescue/rewritten_descriptions.parquet
# ============================================================================

# --- Imports da biblioteca padrão ---
import json
import time
from pathlib import Path

# --- Imports de terceiros ---
import numpy as np
import pandas as pd
import requests

# --- Configuração de display do pandas ---
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)

# --- Detetor defensivo de PROJECT_ROOT (Lição NB06 #1) ---
# Procura ascendentemente por marcadores conhecidos do projeto.
def find_project_root(start: Path, markers=("data", "notebooks", "results")) -> Path:
    """Sobe a árvore de diretórios até encontrar uma pasta que contenha
    todos os marcadores do projeto. Falha de forma explícita se não encontrar."""
    current = start.resolve()
    for parent in [current, *current.parents]:
        if all((parent / m).exists() for m in markers):
            return parent
    raise FileNotFoundError(
        f"Não consegui localizar PROJECT_ROOT a partir de {start}. "
        f"Marcadores procurados: {markers}"
    )

PROJECT_ROOT = find_project_root(Path.cwd())
print(f"PROJECT_ROOT resolvido para: {PROJECT_ROOT}")

# --- Paths do projeto ---
NB06_DIR = PROJECT_ROOT / "results" / "nb06_multimodal"
NB07_DIR = PROJECT_ROOT / "results" / "nb07_llm_rescue"
NB07_DIR.mkdir(parents=True, exist_ok=True)

PRIORITY_QUEUE_PATH = NB06_DIR / "priority_queue.parquet"
assert PRIORITY_QUEUE_PATH.exists(), f"Falta o ficheiro: {PRIORITY_QUEUE_PATH}"
print(f"Priority queue encontrada: {PRIORITY_QUEUE_PATH}")

# --- Configuração do Ollama ---
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "gemma3:4b"

def check_ollama_health(base_url: str, model_name: str) -> dict:
    """Verifica que o servidor Ollama está acessível e que o modelo
    pretendido está disponível localmente. Devolve metadados úteis."""
    # 1. Servidor responde?
    try:
        r = requests.get(f"{base_url}/api/tags", timeout=5)
        r.raise_for_status()
    except requests.exceptions.ConnectionError:
        raise RuntimeError(
            f"Ollama não está a responder em {base_url}. "
            "Inicia o serviço com 'ollama serve' (ou abre a app)."
        )

    # 2. Modelo está instalado?
    available_models = [m["name"] for m in r.json().get("models", [])]
    if model_name not in available_models:
        raise RuntimeError(
            f"Modelo '{model_name}' não encontrado. "
            f"Disponíveis: {available_models}. "
            f"Instala com: ollama pull {model_name}"
        )

    return {
        "server": base_url,
        "model": model_name,
        "all_available_models": available_models,
    }

ollama_status = check_ollama_health(OLLAMA_BASE_URL, OLLAMA_MODEL)
print(f"\n✓ Ollama operacional")
print(f"  Servidor: {ollama_status['server']}")
print(f"  Modelo escolhido: {ollama_status['model']}")
print(f"  Modelos disponíveis no sistema: {len(ollama_status['all_available_models'])}")

# --- Reproducibilidade ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"\nSetup concluído. Output dir: {NB07_DIR}")

PROJECT_ROOT resolvido para: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder
Priority queue encontrada: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb06_multimodal/priority_queue.parquet

✓ Ollama operacional
  Servidor: http://localhost:11434
  Modelo escolhido: gemma3:4b
  Modelos disponíveis no sistema: 5

Setup concluído. Output dir: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb07_llm_rescue


In [2]:
# ============================================================================
# Célula 2 - Carregar priority queue e inspecionar a estrutura
# ============================================================================
# Objetivo: carregar o output do NB06, perceber as colunas disponíveis,
# e validar que temos os dados necessários para o NB07:
#   - PetID (chave)
#   - tier (filtramos High)
#   - Description (texto a reescrever)
#   - Metadados para o prompt (Name, Age, Type, Breed, Gender, etc.)
# ============================================================================

# --- Carregar a priority queue ---
pq = pd.read_parquet(PRIORITY_QUEUE_PATH)

print(f"Shape: {pq.shape}")
print(f"\nColunas disponíveis ({len(pq.columns)}):")
for col in pq.columns:
    print(f"  - {col}: {pq[col].dtype}")

# --- Distribuição por tier ---
print(f"\nDistribuição por tier:")
print(pq["tier"].value_counts())

# --- Primeiras linhas (visão geral) ---
print(f"\nPrimeiras 3 linhas:")
pq.head(3)

Shape: (2997, 21)

Colunas disponíveis (21):
  - rank: int64
  - PetID: str
  - tier: str
  - proba_slow: float64
  - y_proba: float64
  - y_true: int64
  - Name: str
  - Type: int64
  - Age: int64
  - Breed1: int64
  - Gender: int64
  - Color1: int64
  - MaturitySize: int64
  - FurLength: int64
  - Vaccinated: int64
  - Sterilized: int64
  - Health: int64
  - Quantity: int64
  - Fee: int64
  - PhotoAmt: float64
  - Description: str

Distribuição por tier:
tier
Low       1609
Medium     770
High       618
Name: count, dtype: int64

Primeiras 3 linhas:


,rank,PetID,tier,proba_slow,y_proba,y_true,Name,Type,Age,Breed1,Gender,Color1,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Quantity,Fee,PhotoAmt,Description
0,1,e9eeadf82,High,0.525504,0.474496,0,Flash,1,24,307,2,2,2,1,1,1,2,1,20,2.0,Met with an accident and fractured 1 leg. She has healed completely long time ago but this limb ...
1,2,745fd19b8,High,0.525194,0.474806,0,Pooja,1,24,307,2,1,2,1,1,1,2,1,0,3.0,Pooja was founded in a horrible state. She was a friendly dog and used to hang around a taman bu...
2,3,9c3f9ca37,High,0.525169,0.474831,0,Pretty Lady,1,24,307,2,2,2,1,2,1,1,1,0,2.0,"Pretty Lady was an abandoned puppy, and has turned out to be a sweet dog who looks like an Austr..."


In [3]:
# ============================================================================
# Célula 3 - Mapear códigos numéricos para strings legíveis
# ============================================================================
# Contexto: o dataset PetFinder.my codifica Type, Gender, etc. como inteiros.
# Para o LLM gerar texto natural, precisamos de strings ("Dog", "Male",
# "Mixed Breed"). Carregamos os CSVs de labels (Breed, Color) e definimos
# manualmente os mapeamentos do schema oficial Kaggle (os outros campos).
#
# Output: dataframe `pq_enriched` com colunas adicionais sufixadas com `_str`.
# ============================================================================

# --- Mapas do schema oficial Kaggle (PetFinder.my Adoption Prediction) ---
# Fonte: https://www.kaggle.com/c/petfinder-adoption-prediction/data
TYPE_MAP = {1: "Dog", 2: "Cat"}
GENDER_MAP = {1: "Male", 2: "Female", 3: "Mixed"}  # 3 = grupo misto
MATURITY_SIZE_MAP = {
    0: "Not Specified", 1: "Small", 2: "Medium",
    3: "Large", 4: "Extra Large",
}
FUR_LENGTH_MAP = {0: "Not Specified", 1: "Short", 2: "Medium", 3: "Long"}
VACCINATED_MAP = {1: "Yes", 2: "No", 3: "Not Sure"}
STERILIZED_MAP = {1: "Yes", 2: "No", 3: "Not Sure"}
HEALTH_MAP = {
    0: "Not Specified", 1: "Healthy",
    2: "Minor Injury", 3: "Serious Injury",
}

# --- Carregar labels externos (Breed e Color têm muitos valores) ---
breed_labels = pd.read_csv(PROJECT_ROOT / "data" / "breed_labels.csv")
color_labels = pd.read_csv(PROJECT_ROOT / "data" / "color_labels.csv")

print(f"breed_labels: {breed_labels.shape}, colunas: {list(breed_labels.columns)}")
print(f"color_labels: {color_labels.shape}, colunas: {list(color_labels.columns)}")

# Construir dicionários BreedID -> BreedName e ColorID -> ColorName
BREED_MAP = dict(zip(breed_labels["BreedID"], breed_labels["BreedName"]))
COLOR_MAP = dict(zip(color_labels["ColorID"], color_labels["ColorName"]))
# Edge case: Breed1 == 0 significa "no breed specified" no schema
BREED_MAP[0] = "Mixed Breed / Unspecified"

print(f"\n✓ {len(BREED_MAP)} breeds mapeadas")
print(f"✓ {len(COLOR_MAP)} cores mapeadas")

# --- Aplicar mapeamentos ao dataframe ---
pq_enriched = pq.copy()
pq_enriched["Type_str"] = pq_enriched["Type"].map(TYPE_MAP)
pq_enriched["Gender_str"] = pq_enriched["Gender"].map(GENDER_MAP)
pq_enriched["Breed1_str"] = pq_enriched["Breed1"].map(BREED_MAP)
pq_enriched["Color1_str"] = pq_enriched["Color1"].map(COLOR_MAP)
pq_enriched["MaturitySize_str"] = pq_enriched["MaturitySize"].map(MATURITY_SIZE_MAP)
pq_enriched["FurLength_str"] = pq_enriched["FurLength"].map(FUR_LENGTH_MAP)
pq_enriched["Vaccinated_str"] = pq_enriched["Vaccinated"].map(VACCINATED_MAP)
pq_enriched["Sterilized_str"] = pq_enriched["Sterilized"].map(STERILIZED_MAP)
pq_enriched["Health_str"] = pq_enriched["Health"].map(HEALTH_MAP)

# --- Validação: nenhum NaN nos novos campos ---
new_cols = [c for c in pq_enriched.columns if c.endswith("_str")]
nan_counts = pq_enriched[new_cols].isna().sum()
print(f"\nNaN nos campos mapeados (deve ser 0 em todos):")
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else "  ✓ Nenhum NaN — todos os códigos foram mapeados.")

# --- Preview de um animal do tier High (com tudo legível) ---
sample_high = pq_enriched[pq_enriched["tier"] == "High"].iloc[0]
print(f"\n--- Preview de animal do tier High (rank #{sample_high['rank']}) ---")
print(f"  PetID: {sample_high['PetID']}")
print(f"  Name: {sample_high['Name']}")
print(f"  Type: {sample_high['Type_str']}")
print(f"  Age: {sample_high['Age']} months")
print(f"  Breed: {sample_high['Breed1_str']}")
print(f"  Gender: {sample_high['Gender_str']}")
print(f"  Color: {sample_high['Color1_str']}")
print(f"  Size: {sample_high['MaturitySize_str']}")
print(f"  Fur: {sample_high['FurLength_str']}")
print(f"  Vaccinated: {sample_high['Vaccinated_str']}")
print(f"  Sterilized: {sample_high['Sterilized_str']}")
print(f"  Health: {sample_high['Health_str']}")
print(f"\n  Description original:")
print(f"    {sample_high['Description'][:300]}{'...' if len(sample_high['Description']) > 300 else ''}")

breed_labels: (307, 3), colunas: ['BreedID', 'Type', 'BreedName']
color_labels: (7, 2), colunas: ['ColorID', 'ColorName']

✓ 308 breeds mapeadas
✓ 7 cores mapeadas

NaN nos campos mapeados (deve ser 0 em todos):
  ✓ Nenhum NaN — todos os códigos foram mapeados.

--- Preview de animal do tier High (rank #1) ---
  PetID: e9eeadf82
  Name: Flash
  Type: Dog
  Age: 24 months
  Breed: Mixed Breed
  Gender: Female
  Color: Brown
  Size: Medium
  Fur: Short
  Vaccinated: Yes
  Sterilized: Yes
  Health: Minor Injury

  Description original:
    Met with an accident and fractured 1 leg. She has healed completely long time ago but this limb is dysfunctional. Looking for a forever loving home.


In [5]:
# ============================================================================
# Célula 4 - Análise exploratória das descrições do tier High
# ============================================================================
# Objetivo: caracterizar as 618 descrições do tier High antes de fazer
# sampling, para perceber:
#   - Distribuição de comprimentos (palavras e caracteres)
#   - Quantas têm <5 palavras (excluídas — alvo de geração ex-nihilo, NB07b)
#   - Idioma predominante
#   - Casos extremos (muito curtas, muito longas, code-switching)
# ============================================================================

# --- Filtrar tier High ---
high_tier = pq_enriched[pq_enriched["tier"] == "High"].copy()
print(f"Total tier High: {len(high_tier)}")

# --- Lidar com descrições nulas ou vazias ---
# Description pode ser string vazia, NaN, ou só whitespace.
# Tratamos tudo como "vazio" para efeitos de filtragem.
high_tier["Description_clean"] = (
    high_tier["Description"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# --- Métricas de comprimento ---
high_tier["n_words"] = high_tier["Description_clean"].str.split().str.len()
high_tier["n_chars"] = high_tier["Description_clean"].str.len()

# --- Distribuição de palavras ---
print(f"\nDistribuição de palavras nas descrições:")
print(high_tier["n_words"].describe().round(1))

# --- Bucketização por comprimento ---
bins = [-1, 0, 4, 9, 19, 49, 99, np.inf]
labels = ["0 (vazia)", "1-4 (insuf.)", "5-9", "10-19", "20-49", "50-99", "100+"]
high_tier["len_bucket"] = pd.cut(high_tier["n_words"], bins=bins, labels=labels)

print(f"\nDistribuição por bucket de comprimento:")
bucket_counts = high_tier["len_bucket"].value_counts().sort_index()
for label, count in bucket_counts.items():
    pct = 100 * count / len(high_tier)
    print(f"  {label:15s}: {count:4d} ({pct:5.1f}%)")

# --- Candidatos para reescrita (≥5 palavras) ---
candidates = high_tier[high_tier["n_words"] >= 5].copy()
n_candidates = len(candidates)
n_excluded = len(high_tier) - n_candidates
print(f"\n✓ Candidatos a reescrita (≥5 palavras): {n_candidates}")
print(f"✗ Excluídos (geração ex-nihilo, fora de escopo): {n_excluded}")

# --- Casos extremos: 3 mais curtas e 3 mais longas (entre candidatos) ---
print(f"\n--- 3 descrições mais curtas (entre candidatos) ---")
shortest = candidates.nsmallest(3, "n_words")[["PetID", "n_words", "Description_clean"]]
for _, row in shortest.iterrows():
    print(f"  [{row['n_words']} palavras] {row['PetID']}: {row['Description_clean']}")

print(f"\n--- 3 descrições mais longas (entre candidatos) ---")
longest = candidates.nlargest(3, "n_words")[["PetID", "n_words", "Description_clean"]]
for _, row in longest.iterrows():
    preview = row["Description_clean"][:200] + ("..." if len(row["Description_clean"]) > 200 else "")
    print(f"  [{row['n_words']} palavras] {row['PetID']}:")
    print(f"    {preview}")

# --- Sample aleatório de 5 para inspeção qualitativa de idioma e estilo ---
print(f"\n--- 5 descrições aleatórias (inspeção de idioma/estilo) ---")
sample_5 = candidates.sample(5, random_state=RANDOM_SEED)[
    ["PetID", "Type_str", "n_words", "Description_clean"]
]
for _, row in sample_5.iterrows():
    preview = row["Description_clean"][:200] + ("..." if len(row["Description_clean"]) > 200 else "")
    print(f"\n  [{row['Type_str']}, {row['n_words']} palavras] {row['PetID']}:")
    print(f"    {preview}")

Total tier High: 618

Distribuição de palavras nas descrições:
count    618.0
mean      76.9
std       84.2
min        0.0
25%       22.0
50%       47.0
75%       96.8
max      435.0
Name: n_words, dtype: float64

Distribuição por bucket de comprimento:
  0 (vazia)      :    1 (  0.2%)
  1-4 (insuf.)   :   26 (  4.2%)
  5-9            :   42 (  6.8%)
  10-19          :   70 ( 11.3%)
  20-49          :  183 ( 29.6%)
  50-99          :  149 ( 24.1%)
  100+           :  147 ( 23.8%)

✓ Candidatos a reescrita (≥5 palavras): 591
✗ Excluídos (geração ex-nihilo, fora de escopo): 27

--- 3 descrições mais curtas (entre candidatos) ---
  [5 palavras] 949fbd095: 請求一個溫暖的家....... 他是超级活跃的小坏蛋，要有耐心教导的人。。。 有人可以給孩子們一個家嗎？ 有養狗經驗者優先。......... Whatsapp
  [5 palavras] ab2c124e7: Well behaved, playful and healthy
  [5 palavras] b2937acb6: Shes very playfull and smart

--- 3 descrições mais longas (entre candidatos) ---
  [435 palavras] 1c4afcc03:
    18/9/10 Boris threw his wired looped snare, starvation and

In [6]:
# ============================================================================
# Célula 5 - Sampling estratificado de 80 animais para reescrita
# ============================================================================
# Problema: o topo do ranking do NB06 é homogéneo — cães adultos mixed breed
# dominam. Amostragem aleatória ou top-80 daria batch pouco diverso e
# enfraqueceria a demo ao professor.
#
# Solução: sampling estratificado por Type × faixa etária, com proporções
# equilibradas para garantir diversidade sem distorcer demasiado o ranking.
#
# Target: 80 animais com:
#   - ~50% cães / ~50% gatos (corrige desequilíbrio 83/17 do tier High)
#   - 3 faixas etárias equilibradas dentro de cada Type
#   - Prioridade a ranks mais altos dentro de cada estrato (ordem do NB06)
# ============================================================================

N_SAMPLE = 80

# --- Preparar população de candidatos (591 animais ≥5 palavras) ---
pool = candidates.copy()  # vem da Célula 4

# --- Criar faixas etárias (meses) ---
# Bins definidos considerando o dataset: mediana ~12 meses
def age_bucket(months):
    if months <= 6:
        return "puppy_kitten"   # 0-6 meses
    elif months <= 24:
        return "young_adult"    # 7-24 meses
    else:
        return "adult_senior"   # 25+ meses

pool["age_group"] = pool["Age"].apply(age_bucket)

# --- Diagnóstico da população ---
print("Distribuição atual no pool de 591 candidatos:")
cross = pd.crosstab(pool["Type_str"], pool["age_group"], margins=True)
print(cross)
print(f"\nProporção Dog/Cat no pool: {(pool['Type_str']=='Dog').mean():.1%} / {(pool['Type_str']=='Cat').mean():.1%}")

# --- Definir quotas alvo ---
# 40 cães + 40 gatos, distribuídos em 3 faixas etárias (13-14 cada)
QUOTAS = {
    ("Dog", "puppy_kitten"): 13,
    ("Dog", "young_adult"): 14,
    ("Dog", "adult_senior"): 13,
    ("Cat", "puppy_kitten"): 13,
    ("Cat", "young_adult"): 14,
    ("Cat", "adult_senior"): 13,
}
assert sum(QUOTAS.values()) == N_SAMPLE

# --- Sampling dentro de cada estrato ---
# Estratégia: dentro de cada (Type, age_group), ordenar por rank (ascendente
# = maior prioridade do NB06) e pegar os top-K. Assim preservamos o sinal do
# modelo dentro do estrato mas garantimos diversidade entre estratos.
selected_parts = []
diagnostics = []

for (type_str, age_group), quota in QUOTAS.items():
    stratum = pool[
        (pool["Type_str"] == type_str)
        & (pool["age_group"] == age_group)
    ].sort_values("rank", ascending=True)  # rank baixo = mais prioritário

    available = len(stratum)
    taken = min(quota, available)

    if taken < quota:
        diagnostics.append(
            f"  ⚠️  {type_str}/{age_group}: pedidos {quota}, disponíveis {available}"
        )
    else:
        diagnostics.append(
            f"  ✓ {type_str}/{age_group}: {taken}/{quota} (de {available} disponíveis)"
        )

    selected_parts.append(stratum.head(taken))

print(f"\n--- Resultado do sampling ---")
for d in diagnostics:
    print(d)

sample = pd.concat(selected_parts, ignore_index=True)
print(f"\nTotal selecionado: {len(sample)} / {N_SAMPLE}")

# --- Fallback: se algum estrato estava sub-preenchido, completar aleatoriamente ---
shortfall = N_SAMPLE - len(sample)
if shortfall > 0:
    print(f"\n⚠️  Shortfall de {shortfall} animais — completando com sampling aleatório do pool restante")
    remaining = pool[~pool["PetID"].isin(sample["PetID"])]
    extra = remaining.sample(n=shortfall, random_state=RANDOM_SEED)
    sample = pd.concat([sample, extra], ignore_index=True)

# --- Ordenar por rank final para processamento ---
sample = sample.sort_values("rank", ascending=True).reset_index(drop=True)

# --- Validação final ---
print(f"\n--- Composição final do batch ---")
print(pd.crosstab(sample["Type_str"], sample["age_group"], margins=True))

print(f"\nDistribuição de comprimentos no batch selecionado:")
print(sample["n_words"].describe().round(1))

print(f"\nRange de rank no batch: {sample['rank'].min()} — {sample['rank'].max()}")
print(f"(Lembrete: ranks baixos = maior prioridade do NB06)")

# --- Preview dos primeiros 5 selecionados ---
print(f"\n--- Preview: 5 primeiros do batch (por rank) ---")
preview_cols = ["rank", "PetID", "Type_str", "age_group", "Age", "Breed1_str", "n_words"]
print(sample[preview_cols].head(5).to_string(index=False))

Distribuição atual no pool de 591 candidatos:
age_group  adult_senior  puppy_kitten  young_adult  All
Type_str                                               
Cat                  18            25           62  105
Dog                  65           164          257  486
All                  83           189          319  591

Proporção Dog/Cat no pool: 82.2% / 17.8%

--- Resultado do sampling ---
  ✓ Dog/puppy_kitten: 13/13 (de 164 disponíveis)
  ✓ Dog/young_adult: 14/14 (de 257 disponíveis)
  ✓ Dog/adult_senior: 13/13 (de 65 disponíveis)
  ✓ Cat/puppy_kitten: 13/13 (de 25 disponíveis)
  ✓ Cat/young_adult: 14/14 (de 62 disponíveis)
  ✓ Cat/adult_senior: 13/13 (de 18 disponíveis)

Total selecionado: 80 / 80

--- Composição final do batch ---
age_group  adult_senior  puppy_kitten  young_adult  All
Type_str                                               
Cat                  13            13           14   40
Dog                  13            13           14   40
All                  26   

In [7]:
# ============================================================================
# Célula 6 - Inspeção do primeiro animal do batch (caso de teste do prompt)
# ============================================================================
# Objetivo: ver em detalhe o rank #1 para usar como caso de teste na
# engenharia de prompt da próxima célula. Imprimir todos os metadados que
# vão alimentar o prompt + descrição original completa.
# ============================================================================

test_animal = sample.iloc[0]

print("=" * 70)
print(f"ANIMAL DE TESTE — rank #{test_animal['rank']} / PetID {test_animal['PetID']}")
print("=" * 70)

# --- Metadados que vão entrar no prompt ---
print("\nMetadados:")
print(f"  Name:         {test_animal['Name'] if test_animal['Name'] else '(sem nome)'}")
print(f"  Type:         {test_animal['Type_str']}")
print(f"  Breed:        {test_animal['Breed1_str']}")
print(f"  Age:          {test_animal['Age']} months")
print(f"  Gender:       {test_animal['Gender_str']}")
print(f"  Color:        {test_animal['Color1_str']}")
print(f"  Size:         {test_animal['MaturitySize_str']}")
print(f"  Fur length:   {test_animal['FurLength_str']}")
print(f"  Vaccinated:   {test_animal['Vaccinated_str']}")
print(f"  Sterilized:   {test_animal['Sterilized_str']}")
print(f"  Health:       {test_animal['Health_str']}")
print(f"  Quantity:     {test_animal['Quantity']}")
print(f"  Fee:          {test_animal['Fee']} (MYR)")
print(f"  Photos:       {test_animal['PhotoAmt']}")

# --- Predição do modelo NB06 ---
print(f"\nPredição NB06:")
print(f"  proba_slow:   {test_animal['proba_slow']:.4f}")
print(f"  tier:         {test_animal['tier']}")
print(f"  y_true:       {test_animal['y_true']} "
      f"({'genuinamente lento' if test_animal['y_true']==1 else 'foi adotado rápido'})")

# --- Descrição original ---
print(f"\nDescrição original ({test_animal['n_words']} palavras, "
      f"{test_animal['n_chars']} caracteres):")
print("-" * 70)
print(test_animal["Description_clean"])
print("-" * 70)

ANIMAL DE TESTE — rank #1 / PetID e9eeadf82

Metadados:
  Name:         Flash
  Type:         Dog
  Breed:        Mixed Breed
  Age:          24 months
  Gender:       Female
  Color:        Brown
  Size:         Medium
  Fur length:   Short
  Vaccinated:   Yes
  Sterilized:   Yes
  Health:       Minor Injury
  Quantity:     1
  Fee:          20 (MYR)
  Photos:       2.0

Predição NB06:
  proba_slow:   0.5255
  tier:         High
  y_true:       0 (foi adotado rápido)

Descrição original (26 palavras, 148 caracteres):
----------------------------------------------------------------------
Met with an accident and fractured 1 leg. She has healed completely long time ago but this limb is dysfunctional. Looking for a forever loving home.
----------------------------------------------------------------------


In [8]:
# ============================================================================
# Célula 7 - Prompt template + primeira chamada ao Gemma 3
# ============================================================================
# Estrutura do prompt (system + user):
#   SYSTEM: regras invioláveis (não inventar, preservar factos médicos,
#           remover metadata operacional, output em inglês standard,
#           ~80 palavras, só o texto final)
#   USER:   metadados estruturados + descrição original
#
# Primeiro teste: Flash (rank #1, perna disfuncional, 26 palavras de input)
# ============================================================================

# ----------------------------------------------------------------------------
# 7.1 — Construtor de prompt
# ----------------------------------------------------------------------------

SYSTEM_PROMPT = """You are a skilled copywriter for an animal shelter's adoption platform. Your job is to rewrite pet descriptions to be warm, engaging, and clear — increasing the chance adopters will connect with the animal.

STRICT RULES — VIOLATING ANY OF THESE IS A FAILURE:

1. FACTUAL FAITHFULNESS
   - NEVER invent facts not present in the metadata or original description.
   - NEVER omit or soften medical facts (injuries, disabilities, health conditions). If the animal has a disability or injury, name it honestly — an adopter must know what they're signing up for.
   - NEVER change the animal's gender, age, breed, or other attributes.

2. LANGUAGE
   - Output MUST be in clear, standard English — regardless of the input language.
   - Do NOT use "Manglish", chat abbreviations, or regional slang.
   - Do NOT use emojis.

3. CONTENT CLEANUP
   - REMOVE any phone numbers, WhatsApp contacts, email addresses.
   - REMOVE adoption fees, prices, or payment instructions.
   - REMOVE names of volunteers, rescuers, or organizations.
   - REMOVE logistical rules (meeting requirements, home visit policies, etc.).
   - KEEP only content that describes the animal itself: personality, appearance, health, history (when relevant), and what kind of home would suit it.

4. TONE AND LENGTH
   - Warm and engaging, but grounded — not salesy or melodramatic.
   - Target length: approximately 80 words (acceptable range: 60–100).
   - Write in flowing prose, not bullet points.

5. OUTPUT FORMAT
   - Output ONLY the rewritten description — nothing else.
   - Do NOT include preambles like "Here is the rewritten description:".
   - Do NOT offer multiple options or variants.
   - Do NOT ask follow-up questions.
   - Do NOT wrap the output in quotes or markdown."""


def build_user_prompt(row: pd.Series) -> str:
    """Constrói a parte USER do prompt a partir de uma linha do dataframe.

    Inclui metadados estruturados (bullet list) e descrição original.
    Trata ausência de Name e outros campos de forma defensiva.
    """
    name = row["Name"].strip() if row["Name"] and row["Name"].strip() else "(no name given)"

    metadata_block = (
        f"- Name: {name}\n"
        f"- Species: {row['Type_str']}\n"
        f"- Breed: {row['Breed1_str']}\n"
        f"- Age: {row['Age']} months\n"
        f"- Gender: {row['Gender_str']}\n"
        f"- Color: {row['Color1_str']}\n"
        f"- Size: {row['MaturitySize_str']}\n"
        f"- Fur length: {row['FurLength_str']}\n"
        f"- Vaccinated: {row['Vaccinated_str']}\n"
        f"- Sterilized: {row['Sterilized_str']}\n"
        f"- Health status: {row['Health_str']}"
    )

    return (
        "Rewrite the following shelter animal description according to the rules.\n\n"
        "METADATA:\n"
        f"{metadata_block}\n\n"
        "ORIGINAL DESCRIPTION:\n"
        f"{row['Description_clean']}\n\n"
        "REWRITTEN DESCRIPTION:"
    )


# ----------------------------------------------------------------------------
# 7.2 — Função para chamar o Ollama
# ----------------------------------------------------------------------------

def call_ollama(
    system_prompt: str,
    user_prompt: str,
    model: str = OLLAMA_MODEL,
    base_url: str = OLLAMA_BASE_URL,
    temperature: float = 0.4,
    timeout: int = 120,
) -> dict:
    """Chama o endpoint /api/chat do Ollama com system + user prompts.

    Parâmetros:
        temperature: 0.4 = um pouco criativo mas controlado. Demasiado
                     baixo (0.1) dá output rígido; demasiado alto (>0.7)
                     aumenta risco de invenção.
        timeout:     120s cobre descrições longas + primeira chamada
                     (mais lenta por model loading).

    Devolve: dict com {'response': str, 'elapsed_seconds': float, 'raw': dict}
    """
    url = f"{base_url}/api/chat"
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "options": {
            "temperature": temperature,
            # num_predict limita tokens de output (~150 tokens ≈ 100-120 palavras)
            "num_predict": 200,
        },
    }

    start = time.time()
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    elapsed = time.time() - start

    raw = r.json()
    return {
        "response": raw["message"]["content"].strip(),
        "elapsed_seconds": elapsed,
        "raw": raw,
    }


# ----------------------------------------------------------------------------
# 7.3 — Primeiro teste: Flash
# ----------------------------------------------------------------------------

test_animal = sample.iloc[0]
test_user_prompt = build_user_prompt(test_animal)

print("=" * 70)
print("PRIMEIRO TESTE — Flash (rank #1)")
print("=" * 70)
print(f"\n--- System prompt ({len(SYSTEM_PROMPT)} chars) ---")
print(SYSTEM_PROMPT[:300] + "..." if len(SYSTEM_PROMPT) > 300 else SYSTEM_PROMPT)
print(f"\n--- User prompt ({len(test_user_prompt)} chars) ---")
print(test_user_prompt)

print("\n" + "=" * 70)
print("A chamar Gemma 3... (primeira chamada pode demorar 5-15s a carregar o modelo)")
print("=" * 70)

result = call_ollama(SYSTEM_PROMPT, test_user_prompt)

print(f"\n⏱  Tempo: {result['elapsed_seconds']:.2f}s")
print(f"\n--- Descrição original ({test_animal['n_words']} palavras) ---")
print(test_animal["Description_clean"])

n_words_out = len(result["response"].split())
print(f"\n--- Descrição reescrita ({n_words_out} palavras) ---")
print(result["response"])

PRIMEIRO TESTE — Flash (rank #1)

--- System prompt (1748 chars) ---
You are a skilled copywriter for an animal shelter's adoption platform. Your job is to rewrite pet descriptions to be warm, engaging, and clear — increasing the chance adopters will connect with the animal.

STRICT RULES — VIOLATING ANY OF THESE IS A FAILURE:

1. FACTUAL FAITHFULNESS
   - NEVER inve...

--- User prompt (479 chars) ---
Rewrite the following shelter animal description according to the rules.

METADATA:
- Name: Flash
- Species: Dog
- Breed: Mixed Breed
- Age: 24 months
- Gender: Female
- Color: Brown
- Size: Medium
- Fur length: Short
- Vaccinated: Yes
- Sterilized: Yes
- Health status: Minor Injury

ORIGINAL DESCRIPTION:
Met with an accident and fractured 1 leg. She has healed completely long time ago but this limb is dysfunctional. Looking for a forever loving home.

REWRITTEN DESCRIPTION:

A chamar Gemma 3... (primeira chamada pode demorar 5-15s a carregar o modelo)

⏱  Tempo: 6.41s

--- Descrição ori

In [9]:
# ============================================================================
# Célula 7b - Iteração do prompt (v2) + re-teste
# ============================================================================
# Problemas observados na v1 com Flash:
#   1. Inventou "left leg" quando original só dizia "1 leg"
#   2. Suavizou "dysfunctional" para "slightly dysfunctional"
#
# Ambos são sintomas de o LLM adicionar qualificadores/especificidades não
# presentes na fonte. Adicionamos secção explícita contra isto.
# ============================================================================

SYSTEM_PROMPT_V2 = """You are a skilled copywriter for an animal shelter's adoption platform. Your job is to rewrite pet descriptions to be warm, engaging, and clear — increasing the chance adopters will connect with the animal.

STRICT RULES — VIOLATING ANY OF THESE IS A FAILURE:

1. FACTUAL FAITHFULNESS (MOST IMPORTANT)
   - NEVER invent facts not present in the metadata or original description.
   - NEVER omit or soften medical facts (injuries, disabilities, health conditions). If the animal has a disability or injury, name it honestly — an adopter must know what they're signing up for.
   - NEVER change the animal's gender, age, breed, or other attributes.

2. NO ADDED SPECIFICITY OR QUALIFIERS
   - If the original says "1 leg", write "one leg" — do NOT specify "left" or "right".
   - If the original says "dysfunctional", write "dysfunctional" — do NOT soften to "slightly dysfunctional", "mildly affected", or similar.
   - If the original says "injured", do NOT specify severity, location, or cause unless those details are explicitly stated.
   - If a fact is vague in the original, keep it vague in the rewrite. Do not "fill in the gap" with plausible-sounding details.

3. LANGUAGE
   - Output MUST be in clear, standard English — regardless of the input language.
   - Do NOT use Manglish, chat abbreviations, or regional slang.
   - Do NOT use emojis.

4. CONTENT CLEANUP
   - REMOVE any phone numbers, WhatsApp contacts, email addresses.
   - REMOVE adoption fees, prices, or payment instructions.
   - REMOVE names of volunteers, rescuers, or organizations.
   - REMOVE logistical rules (meeting requirements, home visit policies, etc.).
   - KEEP only content that describes the animal itself: personality, appearance, health, history (when relevant), and what kind of home would suit it.

5. TONE AND LENGTH
   - Warm and engaging, but grounded — not salesy or melodramatic.
   - Target length: approximately 80 words (acceptable range: 60–100).
   - Write in flowing prose, not bullet points.

6. OUTPUT FORMAT
   - Output ONLY the rewritten description — nothing else.
   - Do NOT include preambles like "Here is the rewritten description:".
   - Do NOT offer multiple options or variants.
   - Do NOT ask follow-up questions.
   - Do NOT wrap the output in quotes or markdown."""


# ----------------------------------------------------------------------------
# Re-teste com Flash (mesma animal, prompt v2)
# ----------------------------------------------------------------------------
print("=" * 70)
print("TESTE v2 — Flash (mesma animal, prompt iterado)")
print("=" * 70)

result_v2 = call_ollama(SYSTEM_PROMPT_V2, test_user_prompt)

print(f"\n⏱  Tempo: {result_v2['elapsed_seconds']:.2f}s")
print(f"\n--- Descrição original (26 palavras) ---")
print(test_animal["Description_clean"])

n_words_v1 = len(result["response"].split())
print(f"\n--- v1 ({n_words_v1} palavras) — para comparação ---")
print(result["response"])

n_words_v2 = len(result_v2["response"].split())
print(f"\n--- v2 ({n_words_v2} palavras) — prompt iterado ---")
print(result_v2["response"])

# ----------------------------------------------------------------------------
# Segundo teste: caso difícil com metadata operacional
# ----------------------------------------------------------------------------
# Escolher um animal do batch com descrição longa (para testar limpeza de
# metadata operacional + compressão para 80 palavras)
long_candidates = sample[sample["n_words"] >= 150].sort_values("n_words", ascending=False)

if len(long_candidates) > 0:
    test_animal_2 = long_candidates.iloc[0]
    print("\n" + "=" * 70)
    print(f"SEGUNDO TESTE — Caso longo ({test_animal_2['n_words']} palavras)")
    print(f"rank #{test_animal_2['rank']} / PetID {test_animal_2['PetID']}")
    print("=" * 70)

    print(f"\n--- Descrição original ({test_animal_2['n_words']} palavras) ---")
    print(test_animal_2["Description_clean"])

    user_prompt_2 = build_user_prompt(test_animal_2)
    result_long = call_ollama(SYSTEM_PROMPT_V2, user_prompt_2)

    n_words_long = len(result_long["response"].split())
    print(f"\n--- Reescrita v2 ({n_words_long} palavras, {result_long['elapsed_seconds']:.2f}s) ---")
    print(result_long["response"])
else:
    print("\n(sem animal com ≥150 palavras no batch para teste longo)")

TESTE v2 — Flash (mesma animal, prompt iterado)

⏱  Tempo: 2.78s

--- Descrição original (26 palavras) ---
Met with an accident and fractured 1 leg. She has healed completely long time ago but this limb is dysfunctional. Looking for a forever loving home.

--- v1 (71 palavras) — para comparação ---
Flash is a sweet 24-month-old female mixed-breed dog with short, brown fur and a medium size. She experienced a past injury, fracturing her left leg which has fully healed, but the limb remains slightly dysfunctional. Despite this, Flash is a loving companion eager to find a forever home where she’ll receive gentle care and understanding. She’s fully vaccinated and sterilized, and ready to bring joy to a patient and kind adopter.

--- v2 (56 palavras) — prompt iterado ---
Flash is a medium-sized, 24-month-old female mixed-breed dog with short brown fur. She sustained an injury resulting in a fractured one leg, which has healed completely. Despite this, the limb remains dysfunctional. Flash i

In [10]:
# ============================================================================
# Célula 8 - Batch processing: reescrita dos 80 animais
# ============================================================================
# Objetivo: correr o prompt v2 sobre os 80 animais do batch estratificado.
#
# Mecanismos defensivos:
#   - tqdm para progress bar (feedback visual)
#   - Try/except por chamada — um erro não aborta o batch
#   - Checkpoint a cada 20 animais (parquet parcial no disco)
#   - Métricas por chamada (tempo, n_words, erro)
#   - Tempo estimado: 80 × ~5s = ~7 min
# ============================================================================

from tqdm.notebook import tqdm  # progress bar inline no Jupyter

# --- Preparar estrutura de resultados ---
results_list = []
CHECKPOINT_EVERY = 20
CHECKPOINT_PATH = NB07_DIR / "rewritten_descriptions_checkpoint.parquet"

# --- Timestamp de início (para estimativas) ---
batch_start = time.time()
print(f"Início do batch: {time.strftime('%H:%M:%S')}")
print(f"Total a processar: {len(sample)} animais")
print(f"Modelo: {OLLAMA_MODEL} | temperature=0.4")
print(f"Checkpoint a cada {CHECKPOINT_EVERY} animais → {CHECKPOINT_PATH.name}")
print("=" * 70)

# --- Loop principal ---
for i, row in enumerate(tqdm(sample.itertuples(index=False), total=len(sample), desc="Reescrevendo")):
    # Converter namedtuple de volta para Series para usar build_user_prompt
    row_dict = row._asdict()
    row_series = pd.Series(row_dict)

    record = {
        "PetID": row_dict["PetID"],
        "rank": row_dict["rank"],
        "Type_str": row_dict["Type_str"],
        "age_group": row_dict["age_group"],
        "Age": row_dict["Age"],
        "n_words_original": row_dict["n_words"],
        "description_original": row_dict["Description_clean"],
        "description_rewritten": None,
        "n_words_rewritten": None,
        "elapsed_seconds": None,
        "error": None,
    }

    try:
        user_prompt = build_user_prompt(row_series)
        result = call_ollama(SYSTEM_PROMPT_V2, user_prompt)

        record["description_rewritten"] = result["response"]
        record["n_words_rewritten"] = len(result["response"].split())
        record["elapsed_seconds"] = result["elapsed_seconds"]

    except requests.exceptions.Timeout:
        record["error"] = "timeout"
    except requests.exceptions.RequestException as e:
        record["error"] = f"request_error: {str(e)[:200]}"
    except Exception as e:
        record["error"] = f"unexpected: {type(e).__name__}: {str(e)[:200]}"

    results_list.append(record)

    # --- Checkpoint parcial ---
    if (i + 1) % CHECKPOINT_EVERY == 0:
        partial_df = pd.DataFrame(results_list)
        partial_df.to_parquet(CHECKPOINT_PATH, index=False)
        elapsed = time.time() - batch_start
        remaining = len(sample) - (i + 1)
        eta_s = (elapsed / (i + 1)) * remaining
        tqdm.write(
            f"  ↻ Checkpoint {i+1}/{len(sample)} guardado | "
            f"decorrido {elapsed:.0f}s | ETA {eta_s:.0f}s"
        )

# --- Fim do batch ---
batch_elapsed = time.time() - batch_start
print("=" * 70)
print(f"Batch concluído em {batch_elapsed:.1f}s ({batch_elapsed/60:.1f} min)")

# --- Construir dataframe final ---
rewritten_df = pd.DataFrame(results_list)

# --- Diagnóstico rápido ---
n_success = rewritten_df["error"].isna().sum()
n_errors = rewritten_df["error"].notna().sum()

print(f"\n--- Resumo ---")
print(f"  Sucessos: {n_success} / {len(rewritten_df)}")
print(f"  Erros:    {n_errors} / {len(rewritten_df)}")

if n_errors > 0:
    print(f"\n--- Erros observados ---")
    error_summary = rewritten_df[rewritten_df["error"].notna()][["PetID", "rank", "error"]]
    print(error_summary.to_string(index=False))

if n_success > 0:
    success_df = rewritten_df[rewritten_df["error"].isna()]
    print(f"\n--- Estatísticas (só sucessos) ---")
    print(f"  Tempo/animal (s):     mean={success_df['elapsed_seconds'].mean():.2f}, "
          f"min={success_df['elapsed_seconds'].min():.2f}, "
          f"max={success_df['elapsed_seconds'].max():.2f}")
    print(f"  n_words original:     mean={success_df['n_words_original'].mean():.1f}, "
          f"min={success_df['n_words_original'].min()}, "
          f"max={success_df['n_words_original'].max()}")
    print(f"  n_words reescrita:    mean={success_df['n_words_rewritten'].mean():.1f}, "
          f"min={success_df['n_words_rewritten'].min()}, "
          f"max={success_df['n_words_rewritten'].max()}")

    # --- Flagging de outliers de comprimento ---
    too_short = success_df[success_df["n_words_rewritten"] < 40]
    too_long = success_df[success_df["n_words_rewritten"] > 120]
    print(f"\n  Outliers de comprimento (fora 40-120 palavras):")
    print(f"    Muito curtas (<40): {len(too_short)}")
    print(f"    Muito longas (>120): {len(too_long)}")

# --- Guardar output final ---
FINAL_PATH = NB07_DIR / "rewritten_descriptions.parquet"
rewritten_df.to_parquet(FINAL_PATH, index=False)
print(f"\n✓ Output final guardado: {FINAL_PATH}")


Início do batch: 08:05:07
Total a processar: 80 animais
Modelo: gemma3:4b | temperature=0.4
Checkpoint a cada 20 animais → rewritten_descriptions_checkpoint.parquet


Reescrevendo:   0%|          | 0/80 [00:00<?, ?it/s]

  ↻ Checkpoint 20/80 guardado | decorrido 48s | ETA 145s
  ↻ Checkpoint 40/80 guardado | decorrido 97s | ETA 97s
  ↻ Checkpoint 60/80 guardado | decorrido 139s | ETA 46s
  ↻ Checkpoint 80/80 guardado | decorrido 185s | ETA 0s
Batch concluído em 184.5s (3.1 min)

--- Resumo ---
  Sucessos: 80 / 80
  Erros:    0 / 80

--- Estatísticas (só sucessos) ---
  Tempo/animal (s):     mean=2.30, min=1.39, max=3.86
  n_words original:     mean=78.5, min=7, max=381
  n_words reescrita:    mean=61.9, min=35, max=102

  Outliers de comprimento (fora 40-120 palavras):
    Muito curtas (<40): 5
    Muito longas (>120): 0

✓ Output final guardado: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb07_llm_rescue/rewritten_descriptions.parquet


In [11]:
# ============================================================================
# Célula 9 - Avaliação qualitativa: tabela lado-a-lado + inspeção de outliers
# ============================================================================
# Objetivo: produzir a comparação original vs reescrita em formato legível,
# inspecionar os 5 outliers curtos (<40 palavras), e gerar métricas
# agregadas para o relatório.
#
# Prioridade: inspeção visual. Métricas quantitativas vêm depois.
# ============================================================================

# --- Carregar do parquet (para garantir que estamos a ler o artefacto final) ---
df = pd.read_parquet(FINAL_PATH)
success = df[df["error"].isna()].copy()

# --- Métricas agregadas ---
print("=" * 70)
print("MÉTRICAS AGREGADAS")
print("=" * 70)

# Rácio de compressão (n_words_rewritten / n_words_original)
success["compression_ratio"] = success["n_words_rewritten"] / success["n_words_original"]

print(f"\nRácio de compressão (reescrita / original):")
print(f"  Mean:   {success['compression_ratio'].mean():.2f}")
print(f"  Median: {success['compression_ratio'].median():.2f}")
print(f"  Min:    {success['compression_ratio'].min():.2f} "
      f"(original tinha {success.loc[success['compression_ratio'].idxmin(), 'n_words_original']} palavras)")
print(f"  Max:    {success['compression_ratio'].max():.2f} "
      f"(original tinha {success.loc[success['compression_ratio'].idxmax(), 'n_words_original']} palavras)")

# Bucketizar originais para ver como o modelo lida com diferentes comprimentos
def original_bucket(n):
    if n < 20: return "1. Very short (<20)"
    elif n < 50: return "2. Short (20-49)"
    elif n < 100: return "3. Medium (50-99)"
    else: return "4. Long (100+)"

success["original_bucket"] = success["n_words_original"].apply(original_bucket)

print(f"\nComportamento por bucket de input:")
bucket_stats = success.groupby("original_bucket").agg(
    n=("PetID", "count"),
    orig_mean=("n_words_original", "mean"),
    rewrite_mean=("n_words_rewritten", "mean"),
    compression=("compression_ratio", "mean"),
).round(2)
print(bucket_stats)

# --- Inspeção dos outliers curtos (<40 palavras) ---
print("\n" + "=" * 70)
print("INSPEÇÃO DOS 5 OUTLIERS CURTOS (<40 palavras na reescrita)")
print("=" * 70)

short_outliers = success[success["n_words_rewritten"] < 40].sort_values("n_words_rewritten")

for idx, (_, row) in enumerate(short_outliers.iterrows(), 1):
    print(f"\n--- Outlier {idx}/5 — {row['PetID']} (rank #{row['rank']}) ---")
    print(f"  Type: {row['Type_str']} | Age group: {row['age_group']} | "
          f"n_words: {row['n_words_original']} → {row['n_words_rewritten']}")
    print(f"\n  ORIGINAL:")
    print(f"    {row['description_original']}")
    print(f"\n  REESCRITA:")
    print(f"    {row['description_rewritten']}")
    print("-" * 70)

MÉTRICAS AGREGADAS

Rácio de compressão (reescrita / original):
  Mean:   1.66
  Median: 1.12
  Min:    0.26 (original tinha 381 palavras)
  Max:    8.14 (original tinha 7 palavras)

Comportamento por bucket de input:
                      n  orig_mean  rewrite_mean  compression
original_bucket                                              
1. Very short (<20)  13      10.54         47.69         4.85
2. Short (20-49)     26      33.62         53.15         1.65
3. Medium (50-99)    19      79.42         64.00         0.82
4. Long (100+)       22     170.86         78.82         0.51

INSPEÇÃO DOS 5 OUTLIERS CURTOS (<40 palavras na reescrita)

--- Outlier 1/5 — 711bebb99 (rank #135) ---
  Type: Dog | Age group: puppy_kitten | n_words: 20 → 35

  ORIGINAL:
    She is very friendly towards other people and loves to eat treats. She is currently staying at the SPCA Penang.

  REESCRITA:
    Daisy is a friendly four-month-old Golden mixed-breed dog. She’s a medium-sized dog with short fur an

In [12]:
# ============================================================================
# Célula 10 - Export final + amostra representativa para o relatório
# ============================================================================
# Objetivos:
#   1. Gerar CSV para Tableau (parquet já existe)
#   2. Criar amostra qualitativa representativa (10 casos lado-a-lado)
#      cobrindo: short/medium/long inputs, cão/gato, filhote/jovem/adulto
#   3. Gerar metrics.json com todas as métricas chave
# ============================================================================

# --- 10.1 Export CSV para Tableau ---
CSV_PATH = NB07_DIR / "rewritten_descriptions.csv"

# Para o Tableau queremos colunas essenciais e legíveis
tableau_cols = [
    "PetID", "rank", "Type_str", "age_group", "Age",
    "n_words_original", "n_words_rewritten",
    "description_original", "description_rewritten",
    "error",
]
df[tableau_cols].to_csv(CSV_PATH, index=False)
print(f"✓ CSV para Tableau: {CSV_PATH}")
print(f"  Shape: {df[tableau_cols].shape}")

# --- 10.2 Amostra qualitativa representativa (10 casos) ---
# Critério: 2 casos por bucket de input + 2 ninhadas (se houver)
sample_cases = []

# 2 casos de cada bucket (escolher diversidade Type)
for bucket in ["1. Very short (<20)", "2. Short (20-49)", "3. Medium (50-99)", "4. Long (100+)"]:
    bucket_df = success[success["original_bucket"] == bucket]
    if len(bucket_df) >= 2:
        # Um Dog, um Cat se possível
        dogs_in_bucket = bucket_df[bucket_df["Type_str"] == "Dog"]
        cats_in_bucket = bucket_df[bucket_df["Type_str"] == "Cat"]

        if len(dogs_in_bucket) > 0:
            sample_cases.append(dogs_in_bucket.sample(1, random_state=RANDOM_SEED).iloc[0])
        if len(cats_in_bucket) > 0:
            sample_cases.append(cats_in_bucket.sample(1, random_state=RANDOM_SEED).iloc[0])

qualitative_sample = pd.DataFrame(sample_cases).drop_duplicates(subset="PetID").head(10)

print(f"\n✓ Amostra qualitativa: {len(qualitative_sample)} casos")

# --- Guardar amostra qualitativa (para relatório) ---
QUAL_PATH = NB07_DIR / "qualitative_sample.csv"
qualitative_sample[tableau_cols].to_csv(QUAL_PATH, index=False)
print(f"✓ Amostra qualitativa guardada: {QUAL_PATH}")

# --- Imprimir amostra formatada para inspeção ---
print("\n" + "=" * 70)
print("AMOSTRA QUALITATIVA (para relatório)")
print("=" * 70)

for idx, (_, row) in enumerate(qualitative_sample.iterrows(), 1):
    print(f"\n--- Caso {idx}/{len(qualitative_sample)} | "
          f"{row['Type_str']} | {row['age_group']} | rank #{row['rank']} ---")
    print(f"[{row['n_words_original']} → {row['n_words_rewritten']} palavras]")
    print(f"\nANTES:")
    print(f"  {row['description_original']}")
    print(f"\nDEPOIS:")
    print(f"  {row['description_rewritten']}")
    print()

# --- 10.3 Metrics consolidadas em JSON ---
METRICS_PATH = NB07_DIR / "metrics.json"

metrics = {
    "batch_info": {
        "model": OLLAMA_MODEL,
        "temperature": 0.4,
        "n_target": N_SAMPLE,
        "n_success": int(n_success),
        "n_errors": int(n_errors),
        "success_rate": float(n_success / len(df)),
        "total_time_seconds": round(batch_elapsed, 1),
        "mean_time_per_animal": round(success["elapsed_seconds"].mean(), 2),
    },
    "prompt_engineering": {
        "n_iterations": 2,
        "v1_issues": [
            "invented specificity (e.g. 'left leg' from '1 leg')",
            "softened medical severity (e.g. 'slightly dysfunctional')",
        ],
        "v2_fix": "added explicit 'NO ADDED SPECIFICITY OR QUALIFIERS' section with concrete examples",
    },
    "length_stats": {
        "original": {
            "mean": round(success["n_words_original"].mean(), 1),
            "median": float(success["n_words_original"].median()),
            "min": int(success["n_words_original"].min()),
            "max": int(success["n_words_original"].max()),
        },
        "rewritten": {
            "mean": round(success["n_words_rewritten"].mean(), 1),
            "median": float(success["n_words_rewritten"].median()),
            "min": int(success["n_words_rewritten"].min()),
            "max": int(success["n_words_rewritten"].max()),
        },
        "compression_ratio": {
            "mean": round(success["compression_ratio"].mean(), 2),
            "median": round(success["compression_ratio"].median(), 2),
        },
    },
    "behavior_by_input_length": {
        bucket: {
            "n": int(row["n"]),
            "original_mean_words": float(row["orig_mean"]),
            "rewritten_mean_words": float(row["rewrite_mean"]),
            "compression": float(row["compression"]),
        }
        for bucket, row in bucket_stats.iterrows()
    },
    "outliers": {
        "n_short_lt40": int(len(short_outliers)),
        "n_long_gt120": 0,
        "short_outliers_notes": "All 5 cases had input <30 words. Model expanded honestly using metadata without inventing facts.",
    },
    "batch_composition": {
        "n_dogs": int((success["Type_str"] == "Dog").sum()),
        "n_cats": int((success["Type_str"] == "Cat").sum()),
        "by_age_group": success["age_group"].value_counts().to_dict(),
    },
    "known_limitations": [
        "Litter coherence: when original mentions multiple animals (Quantity>1), model sometimes singularizes. Prompt does not include Quantity field — addressable in future iteration.",
        "Slight ungrammatical fragments on very short inputs (e.g. 'a fractured one leg') — tradeoff accepted vs risk of fact invention.",
        "Batch covers rank 1-562 of tier High (not only top), chosen to ensure diversity; top-ranked animals could be processed separately if needed.",
    ],
}

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print("=" * 70)
print(f"✓ Métricas guardadas: {METRICS_PATH}")
print("=" * 70)

# --- Resumo final ---
print("\nARTEFACTOS DO NB07:")
for p in sorted(NB07_DIR.glob("*")):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name} ({size_kb:.1f} KB)")

✓ CSV para Tableau: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb07_llm_rescue/rewritten_descriptions.csv
  Shape: (80, 10)

✓ Amostra qualitativa: 8 casos
✓ Amostra qualitativa guardada: /Users/rennandamiani/Documents/ISAG/ProjetoFinal_PetFinder/results/nb07_llm_rescue/qualitative_sample.csv

AMOSTRA QUALITATIVA (para relatório)

--- Caso 1/8 | Dog | puppy_kitten | rank #131 ---
[13 → 48 palavras]

ANTES:
  she is a loving and friendly pet....she is more of a protective dog

DEPOIS:
  Kutty is a medium-sized, five-month-old female dog with short black fur. She is a healthy mixed-breed and has been sterilized. Kutty is a loving and friendly dog who can also be protective. She would thrive in a patient and understanding home where she can feel safe and secure.


--- Caso 2/8 | Cat | puppy_kitten | rank #366 ---
[12 → 49 palavras]

ANTES:
  Found them at my place..juz need adopter because im at training center

DEPOIS:
  Wolf,batman,sally is a small, black, Domest